<a href="https://colab.research.google.com/github/Yash-Yelave/Global_Gatway_RS/blob/main/100news_Scrapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Dependencies

In [1]:
!pip install groq newspaper3k feedparser requests beautifulsoup4 pandas pydantic lxml_html_clean -q

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 35.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 8.9 MB/s eta 0:00:00


Imports & config


In [2]:
import os
import time
import json
import hashlib
import sqlite3
import feedparser
import pandas as pd
import requests

from datetime import datetime
from bs4 import BeautifulSoup
from newspaper import Article
from groq import Groq
from pydantic import BaseModel, field_validator
from typing import Optional, List
from google.colab import userdata

client = Groq(api_key=userdata.get("GROQ_API_KEY_1"))

GROQ_MODEL          = "llama-3.1-8b-instant"
MAX_TOKENS_IN       = 2000
RATE_LIMIT_SEC      = 2.5
MAX_RETRIES         = 3
RETRY_WAIT_SEC      = 65        # wait 65s when quota is hit (resets per minute)
ARTICLES_PER_FEED   = 5
CATEGORIES          = [
    "technology", "politics", "sports", "business",
    "health", "science", "entertainment", "general",
    "finance", "travel", "culture", "environment"
]

100 RSS feed sources (UAE + China heavy)

In [3]:
RSS_FEEDS = {

    # ── UAE / Gulf / Arabic ──────────────────────────────────────────────────
    "Khaleej Times"              : "https://www.khaleejtimes.com/rss",
    "Gulf News"                  : "https://gulfnews.com/rss",
    "The National UAE"           : "https://www.thenationalnews.com/rss",
    "Gulf Business"              : "https://gulfbusiness.com/feed/",
    "Emirates 24/7"              : "https://www.emirates247.com/rss",
    "Arabian Business"           : "https://www.arabianbusiness.com/rss",
    "Al Arabiya English"         : "https://english.alarabiya.net/tools/rss",
    "Al Jazeera English"         : "https://www.aljazeera.com/xml/rss/all.xml",
    "Zawya UAE"                  : "https://www.zawya.com/rss/uae/",
    "Dubai Eye News"             : "https://www.dubaieye1038.com/feed/",
    "Al Khaleej (Arabic)"        : "https://www.alkhaleej.ae/rss.xml",
    "Al Bayan (Arabic)"          : "https://albayan.ae/rss",
    "WAM UAE State News"         : "https://wam.ae/rss.xml",
    "Time Out Dubai"             : "https://www.timeoutdubai.com/rss",
    "What's On Dubai"            : "https://whatson.ae/feed/",
    "Construction Week Online"   : "https://www.constructionweekonline.com/rss",
    "MEED Middle East"           : "https://www.meed.com/rss/",
    "Arab News"                  : "https://www.arabnews.com/rss.xml",
    "Saudi Gazette"              : "https://saudigazette.com.sa/rss",
    "Oman Observer"              : "https://www.omanobserver.om/feed/",
    "Kuwait Times"               : "https://www.kuwaittimes.com/feed/",
    "Bahrain News Agency"        : "https://www.bna.bh/rss.xml",
    "Qatar Tribune"              : "https://www.qatar-tribune.com/rss",
    "Middle East Eye"            : "https://www.middleeasteye.net/rss",
    "Roya News Jordan"           : "https://en.royanews.tv/rss.xml",

    # ── China / Hong Kong / Asia ─────────────────────────────────────────────
    "China Daily"                : "https://www.chinadaily.com.cn/rss/china_rss.xml",
    "China Daily Business"       : "https://www.chinadaily.com.cn/rss/bizChina_rss.xml",
    "CGTN World"                 : "https://www.cgtn.com/subscribe/rss/section/world.xml",
    "CGTN Business"              : "https://www.cgtn.com/subscribe/rss/section/business.xml",
    "CGTN Science"               : "https://www.cgtn.com/subscribe/rss/section/sci-tech.xml",
    "Xinhua Top News"            : "https://www.xinhuanet.com/english/rss/worldrss.xml",
    "Xinhua China"               : "https://www.xinhuanet.com/english/rss/chinarss.xml",
    "South China Morning Post"   : "https://www.scmp.com/rss/2/feed",
    "SCMP Business"              : "https://www.scmp.com/rss/92/feed",
    "SCMP Tech"                  : "https://www.scmp.com/rss/36/feed",
    "Hong Kong Free Press"       : "https://hongkongfp.com/feed/",
    "Caixin Global"              : "https://www.caixinglobal.com/rss/index.xml",
    "Sixth Tone China"           : "https://www.sixthtone.com/rss",
    "The Diplomat Asia"          : "https://thediplomat.com/feed/",
    "Asia Times"                 : "https://asiatimes.com/feed/",
    "Nikkei Asia"                : "https://asia.nikkei.com/rss/feed/nar",
    "Japan Times"                : "https://www.japantimes.co.jp/feed/",
    "Korea Herald"               : "http://www.koreaherald.com/rss/050000000000.xml",
    "Times of India"             : "https://timesofindia.indiatimes.com/rssfeedstopstories.cms",
    "Straits Times Singapore"    : "https://www.straitstimes.com/global/rss.xml",

    # ── International / Global ───────────────────────────────────────────────
    "BBC Top Stories"            : "http://feeds.bbci.co.uk/news/rss.xml",
    "BBC World"                  : "http://feeds.bbci.co.uk/news/world/rss.xml",
    "BBC Technology"             : "http://feeds.bbci.co.uk/news/technology/rss.xml",
    "BBC Business"               : "http://feeds.bbci.co.uk/news/business/rss.xml",
    "BBC Health"                 : "http://feeds.bbci.co.uk/news/health/rss.xml",
    "Reuters Top News"           : "https://feeds.reuters.com/reuters/topNews",
    "Reuters World"              : "https://feeds.reuters.com/Reuters/worldNews",
    "Reuters Business"           : "https://feeds.reuters.com/reuters/businessNews",
    "Reuters Technology"         : "https://feeds.reuters.com/reuters/technologyNews",
    "AP Top News"                : "https://feeds.apnews.com/rss/apf-topnews",
    "AP World"                   : "https://feeds.apnews.com/rss/apf-WorldNews",
    "AP Business"                : "https://feeds.apnews.com/rss/apf-business",
    "The Guardian World"         : "https://www.theguardian.com/world/rss",
    "The Guardian Tech"          : "https://www.theguardian.com/uk/technology/rss",
    "The Guardian Business"      : "https://www.theguardian.com/business/rss",
    "The Guardian Environment"   : "https://www.theguardian.com/environment/rss",

    # ── Technology ───────────────────────────────────────────────────────────
    "TechCrunch"                 : "https://techcrunch.com/feed/",
    "TechCrunch AI"              : "https://techcrunch.com/category/artificial-intelligence/feed/",
    "The Verge"                  : "https://www.theverge.com/rss/index.xml",
    "Wired"                      : "https://www.wired.com/feed/rss",
    "Ars Technica"               : "http://feeds.arstechnica.com/arstechnica/index",
    "MIT Tech Review"            : "https://www.technologyreview.com/feed/",
    "VentureBeat"                : "https://venturebeat.com/feed/",
    "ZDNet"                      : "https://www.zdnet.com/news/rss.xml",
    "Engadget"                   : "https://www.engadget.com/rss.xml",
    "Mashable Tech"              : "https://mashable.com/feeds/rss/tech",

    # ── Business / Finance ───────────────────────────────────────────────────
    "Bloomberg Technology"       : "https://feeds.bloomberg.com/technology/news.rss",
    "CNBC World"                 : "https://www.cnbc.com/id/100727362/device/rss/rss.html",
    "CNBC Finance"               : "https://www.cnbc.com/id/10000664/device/rss/rss.html",
    "Financial Times World"      : "https://www.ft.com/world?format=rss",
    "Forbes Business"            : "https://www.forbes.com/business/feed/",
    "Forbes Tech"                : "https://www.forbes.com/technology/feed/",
    "Business Insider"           : "https://feeds.businessinsider.com/custom/all",
    "MarketWatch"                : "https://feeds.marketwatch.com/marketwatch/topstories/",
    "Investopedia"               : "https://www.investopedia.com/feedbuilder/feed/getfeed?feedName=rss_headline",
    "CoinDesk Crypto"            : "https://www.coindesk.com/arc/outboundfeeds/rss/",

    # ── Health & Science ─────────────────────────────────────────────────────
    "WHO News"                   : "https://www.who.int/rss-feeds/news-english.xml",
    "Harvard Health"             : "https://www.health.harvard.edu/blog/feed",
    "Science Daily"              : "https://www.sciencedaily.com/rss/all.xml",
    "Nature News"                : "https://www.nature.com/nature.rss",
    "NASA Breaking News"         : "https://www.nasa.gov/rss/dyn/breaking_news.rss",
    "New Scientist"              : "https://www.newscientist.com/feed/home/",
    "Medical News Today"         : "https://www.medicalnewstoday.com/rss",

    # ── Sports ───────────────────────────────────────────────────────────────
    "ESPN Headlines"             : "https://www.espn.com/espn/rss/news",
    "BBC Sport"                  : "http://feeds.bbci.co.uk/sport/rss.xml",
    "Sky Sports"                 : "https://www.skysports.com/rss/12040",
    "Goal.com Football"          : "https://www.goal.com/feeds/en/news",
    "Sports Illustrated"         : "https://www.si.com/rss/si_topstories.rss",

    # ── Environment / Travel ─────────────────────────────────────────────────
    "National Geographic"        : "https://www.nationalgeographic.com/news/rss",
    "BBC Travel"                 : "http://feeds.bbci.co.uk/travel/rss.xml",
    "Lonely Planet"              : "https://www.lonelyplanet.com/news/feed",
    "CNN Travel"                 : "http://rss.cnn.com/rss/edition_travel.rss",
    "World Wildlife Fund"        : "https://www.worldwildlife.org/magazine/rss",

    # ── Entertainment / Culture ──────────────────────────────────────────────
    "Variety"                    : "https://variety.com/feed/",
    "Hollywood Reporter"         : "https://www.hollywoodreporter.com/feed/",
    "Rolling Stone"              : "https://www.rollingstone.com/feed/",
    "Pitchfork Music"            : "https://pitchfork.com/rss/news/",
    "Deadline Hollywood"         : "https://deadline.com/feed/",
}

print(f"Total feeds configured: {len(RSS_FEEDS)}")

Total feeds configured: 103


RSS URL fetcher

In [4]:
def fetch_rss_urls(feeds: dict, max_per_feed: int = ARTICLES_PER_FEED) -> list[dict]:
    articles  = []
    failed    = []
    seen_urls = set()

    for source_name, feed_url in feeds.items():
        try:
            feed    = feedparser.parse(feed_url)
            entries = feed.entries[:max_per_feed]
            count   = 0

            for entry in entries:
                url = entry.get("link", "")
                if not url or url in seen_urls:
                    continue
                seen_urls.add(url)
                articles.append({
                    "source"   : source_name,
                    "url"      : url,
                    "rss_title": entry.get("title", ""),
                    "published": entry.get("published", ""),
                })
                count += 1

            status = f"✓ {count} articles" if count else "⚠ 0 articles (feed may be empty)"
            print(f"  {status} — {source_name}")

        except Exception as e:
            failed.append(source_name)
            print(f"  ✗ FAILED — {source_name}: {e}")

    print(f"\nRSS fetch done. {len(articles)} URLs | {len(failed)} feeds failed.")
    if failed:
        print("Failed feeds:", failed)

    return articles

print("Fetching all RSS feeds...\n")
raw_urls = fetch_rss_urls(RSS_FEEDS, max_per_feed=ARTICLES_PER_FEED)
print(f"\nTotal URLs to process: {len(raw_urls)}")

Fetching all RSS feeds...

  ⚠ 0 articles (feed may be empty) — Khaleej Times
  ⚠ 0 articles (feed may be empty) — Gulf News
  ⚠ 0 articles (feed may be empty) — The National UAE
  ⚠ 0 articles (feed may be empty) — Gulf Business
  ⚠ 0 articles (feed may be empty) — Emirates 24/7
  ⚠ 0 articles (feed may be empty) — Arabian Business
  ⚠ 0 articles (feed may be empty) — Al Arabiya English
  ✓ 5 articles — Al Jazeera English
  ⚠ 0 articles (feed may be empty) — Zawya UAE
  ⚠ 0 articles (feed may be empty) — Dubai Eye News
  ⚠ 0 articles (feed may be empty) — Al Khaleej (Arabic)
  ⚠ 0 articles (feed may be empty) — Al Bayan (Arabic)
  ⚠ 0 articles (feed may be empty) — WAM UAE State News
  ⚠ 0 articles (feed may be empty) — Time Out Dubai
  ✓ 5 articles — What's On Dubai
  ⚠ 0 articles (feed may be empty) — Construction Week Online
  ⚠ 0 articles (feed may be empty) — MEED Middle East
  ⚠ 0 articles (feed may be empty) — Arab News
  ⚠ 0 articles (feed may be empty) — Saudi Gazette
  ⚠ 0 a

Article scraper

In [5]:
def scrape_article(url: str) -> dict | None:
    try:
        art = Article(url, fetch_images=False, request_timeout=12)
        art.download()
        art.parse()

        meta_desc = ""
        try:
            soup = BeautifulSoup(art.html or "", "html.parser")
            tag  = (soup.find("meta", attrs={"name": "description"}) or
                    soup.find("meta", attrs={"property": "og:description"}))
            if tag:
                meta_desc = tag.get("content", "")
        except Exception:
            pass

        return {
            "url"           : url,
            "scraped_title" : art.title or "",
            "raw_text"      : art.text or "",
            "scraped_author": ", ".join(art.authors) if art.authors else "",
            "scraped_date"  : str(art.publish_date) if art.publish_date else "",
            "meta_desc"     : meta_desc,
            "top_image"     : art.top_image or "",
        }
    except Exception as e:
        print(f"    ✗ Scrape failed: {e}")
        return None

Groq extraction with full quota handling

In [6]:
EXTRACTION_PROMPT = """
You are a structured data extractor for a news recommendation system.

Given the raw text of a news article, return ONLY a valid JSON object.
No explanation, no markdown fences, no extra text — just the JSON.

JSON schema:
{{
  "title"       : "clean headline (string)",
  "description" : "2-3 sentence plain-English summary written in your own words",
  "author"      : "author full name or null",
  "tags"        : ["keyword1", "keyword2", "keyword3"],
  "category"    : "one of: technology | politics | sports | business | health | science | entertainment | finance | travel | culture | environment | general",
  "sentiment"   : "positive | neutral | negative",
  "publish_date": "YYYY-MM-DD or null",
  "language"    : "en | ar | zh | other"
}}

Rules:
- tags: 3 to 5 short lowercase keywords.
- description: your own summary, never copied text.
- If a field cannot be determined use null.
- Return ONLY the JSON object.

Article title hint: {title_hint}
Article text:
{article_text}
""".strip()


def extract_with_groq(scraped: dict, stats: dict) -> dict | None:
    """
    Call Groq with retry logic and quota exhaustion handling.
    stats dict is mutated in-place to track API usage.
    """
    raw_text = scraped.get("raw_text", "").strip()
    if not raw_text:
        raw_text = scraped.get("meta_desc", "").strip()
    if not raw_text:
        print("    ⚠ No text to extract from — skipping")
        return None

    truncated = raw_text[:MAX_TOKENS_IN * 4]
    prompt    = EXTRACTION_PROMPT.format(
        title_hint   = scraped.get("scraped_title", ""),
        article_text = truncated,
    )

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(
                model       = GROQ_MODEL,
                messages    = [{"role": "user", "content": prompt}],
                max_tokens  = 512,
                temperature = 0.1,
            )
            stats["api_calls"] += 1
            raw_json = response.choices[0].message.content.strip()

            # Strip accidental markdown fences
            if raw_json.startswith("```"):
                raw_json = raw_json.split("```")[1]
                if raw_json.startswith("json"):
                    raw_json = raw_json[4:]

            parsed = json.loads(raw_json)
            return parsed

        except Exception as e:
            err_str = str(e).lower()

            # ── Quota / rate limit hit ────────────────────────────────────────
            if any(kw in err_str for kw in ["rate_limit", "rate limit", "429",
                                             "quota", "too many requests",
                                             "tokens per minute",
                                             "requests per minute"]):
                stats["quota_hits"] += 1
                print(f"\n{'='*60}")
                print(f"  ⚠  GROQ QUOTA HIT (attempt {attempt}/{MAX_RETRIES})")
                print(f"  API calls so far : {stats['api_calls']}")
                print(f"  Articles saved   : {stats['articles_saved']}")
                print(f"  Waiting {RETRY_WAIT_SEC}s for quota reset...")
                print(f"{'='*60}\n")
                time.sleep(RETRY_WAIT_SEC)
                continue

            # ── JSON parse error ──────────────────────────────────────────────
            elif "json" in err_str or isinstance(e, json.JSONDecodeError):
                print(f"    ✗ JSON parse error (attempt {attempt}): {e}")
                if attempt == MAX_RETRIES:
                    return None
                time.sleep(2)
                continue

            # ── Auth error — hard stop ────────────────────────────────────────
            elif any(kw in err_str for kw in ["authentication", "api_key",
                                               "unauthorized", "401"]):
                print("\n✗ GROQ AUTH ERROR — check your API key in Colab Secrets.")
                raise SystemExit("Invalid Groq API key.")

            # ── Generic error ─────────────────────────────────────────────────
            else:
                print(f"    ✗ Groq error (attempt {attempt}): {e}")
                if attempt == MAX_RETRIES:
                    return None
                time.sleep(3)
                continue

    return None

Pydantic schema


In [7]:
class NewsArticle(BaseModel):
    article_id  : str
    url         : str
    source      : str
    title       : str
    description : str
    author      : Optional[str]
    tags        : List[str]
    category    : str
    sentiment   : str
    publish_date: Optional[str]
    language    : Optional[str]
    top_image   : Optional[str]
    scraped_at  : str

    @field_validator("category")
    @classmethod
    def validate_category(cls, v):
        return v if v in CATEGORIES else "general"

    @field_validator("sentiment")
    @classmethod
    def validate_sentiment(cls, v):
        return v if v in ("positive", "neutral", "negative") else "neutral"

    @field_validator("tags")
    @classmethod
    def validate_tags(cls, v):
        return [t.lower().strip() for t in v if t.strip()][:5]

    @field_validator("title", "description")
    @classmethod
    def not_empty(cls, v):
        if not v or not v.strip():
            raise ValueError("Field cannot be empty")
        return v.strip()


def build_article_id(url: str) -> str:
    return hashlib.md5(url.encode()).hexdigest()[:12]


def validate_and_merge(rss_meta: dict, scraped: dict, extracted: dict) -> NewsArticle | None:
    try:
        return NewsArticle(
            article_id  = build_article_id(rss_meta["url"]),
            url         = rss_meta["url"],
            source      = rss_meta["source"],
            title       = (extracted.get("title") or
                           scraped.get("scraped_title") or
                           rss_meta.get("rss_title", "")),
            description = extracted.get("description", ""),
            author      = (extracted.get("author") or
                           scraped.get("scraped_author") or None),
            tags        = extracted.get("tags", []),
            category    = extracted.get("category", "general"),
            sentiment   = extracted.get("sentiment", "neutral"),
            publish_date= (extracted.get("publish_date") or
                           scraped.get("scraped_date") or
                           rss_meta.get("published") or None),
            language    = extracted.get("language", "en"),
            top_image   = scraped.get("top_image") or None,
            scraped_at  = datetime.utcnow().isoformat(),
        )
    except Exception as e:
        print(f"    ✗ Validation error: {e}")
        return None

Save helper

In [8]:
def save_outputs(articles: list[dict], prefix: str = "news_dataset") -> pd.DataFrame:
    df = pd.DataFrame(articles)
    if df.empty:
        print("⚠ No articles to save yet.")
        return df

    # CSV
    csv_path = f"{prefix}.csv"
    df.to_csv(csv_path, index=False)

    # JSON
    json_path = f"{prefix}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(articles, f, indent=2, ensure_ascii=False)

    # SQLite
    db_path  = f"{prefix}.db"
    conn     = sqlite3.connect(db_path)
    df_sql   = df.copy()
    df_sql["tags"] = df_sql["tags"].apply(json.dumps)
    df_sql.to_sql("articles", conn, if_exists="replace", index=False)
    conn.commit()
    conn.close()

    print(f"  💾 Saved {len(df)} articles → {csv_path} | {json_path} | {db_path}")
    return df

Main pipeline

In [9]:
def run_pipeline(raw_urls: list[dict], max_articles: int = 500,
                 autosave_every: int = 25) -> list[dict]:
    """
    Full pipeline with:
      - Groq quota retry handling
      - Live progress reporting
      - Auto-save every N articles so you never lose work
      - Final summary on completion
    """
    results   = []
    seen_urls = set()
    processed = 0
    skipped   = 0

    stats = {
        "api_calls"    : 0,
        "quota_hits"   : 0,
        "articles_saved": 0,
    }

    total_urls = min(len(raw_urls), max_articles * 3)  # rough upper bound
    print(f"Starting pipeline — target: {max_articles} articles from {len(raw_urls)} URLs\n")
    print("=" * 65)

    for idx, item in enumerate(raw_urls):
        if processed >= max_articles:
            break

        url = item["url"]
        if url in seen_urls:
            continue
        seen_urls.add(url)

        print(f"\n[{processed + 1}/{max_articles}] Source: {item['source']}")
        print(f"  URL: {url[:75]}")

        # ── Step 1: Scrape ─────────────────────────────────────────────────
        scraped = scrape_article(url)
        if not scraped:
            skipped += 1
            print("  → Scrape failed, skipping.")
            continue

        # ── Step 2: Groq extraction ────────────────────────────────────────
        extracted = extract_with_groq(scraped, stats)
        if not extracted:
            skipped += 1
            print("  → Groq extraction failed, skipping.")
            continue

        # ── Step 3: Validate & merge ───────────────────────────────────────
        article = validate_and_merge(item, scraped, extracted)
        if not article:
            skipped += 1
            print("  → Validation failed, skipping.")
            continue

        results.append(article.model_dump())
        processed            += 1
        stats["articles_saved"] = processed

        print(f"  ✓ [{article.language}] [{article.category}] [{article.sentiment}]")
        print(f"    Title : {article.title[:70]}")
        print(f"    Author: {article.author or 'N/A'} | Tags: {', '.join(article.tags)}")

        # ── Auto-save checkpoint ───────────────────────────────────────────
        if processed % autosave_every == 0:
            print(f"\n  ── Auto-save checkpoint at {processed} articles ──")
            save_outputs(results)

        time.sleep(RATE_LIMIT_SEC)

    # ── Final save ─────────────────────────────────────────────────────────
    print("\n" + "=" * 65)
    print("PIPELINE COMPLETE")
    print(f"  Articles collected : {processed}")
    print(f"  URLs skipped       : {skipped}")
    print(f"  Groq API calls     : {stats['api_calls']}")
    print(f"  Quota hits         : {stats['quota_hits']}")
    print("=" * 65)
    save_outputs(results)

    return results


# ── Run ─────────────────────────────────────────────────────────────────────
# max_articles = 500 targets 5 per feed across 100 feeds.
# Lower it to 50-100 first to do a test run.
clean_articles = run_pipeline(raw_urls, max_articles=500, autosave_every=25)

Starting pipeline — target: 500 articles from 285 URLs


[1/500] Source: Al Jazeera English
  URL: https://www.aljazeera.com/news/2026/4/21/iran-us-war-four-scenarios-for-wha


/tmp/ipykernel_19745/119334936.py:63: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  scraped_at  = datetime.utcnow().isoformat(),


  ✓ [en] [politics] [neutral]
    Title : Iran-US war: Four scenarios for what’s next as talks stumble
    Author: Yashraj Sharma | Tags: iran, us, war, talks, ceasefire

[2/500] Source: Al Jazeera English
  URL: https://www.aljazeera.com/video/doha-debates/2026/4/21/are-we-heading-into-
  ✓ [en] [technology] [neutral]
    Title : Are we heading into a world divided by AI tribes?
    Author: N/A | Tags: ai, society, technology

[3/500] Source: Al Jazeera English
  URL: https://www.aljazeera.com/news/2026/4/21/what-was-the-iran-nuclear-deal-tru
  ✓ [en] [politics] [neutral]
    Title : What was the Iran nuclear deal Trump dumped in search of ‘better’ term
    Author: Usaid Siddiqui | Tags: iran, nuclear, deal, trump, politics

[4/500] Source: Al Jazeera English
  URL: https://www.aljazeera.com/video/newsfeed/2026/4/21/mass-trial-for-nearly-50
  ✓ [en] [politics] [neutral]
    Title : mass trial for nearly 500 alleged gang members in el salvador
    Author: N/A | Tags: mass trial, el sal

/usr/local/lib/python3.12/dist-packages/dateutil/parser/_parser.py:1207: UnknownTimezoneWarning: tzname PDT identified but not understood.  Pass `tzinfos` argument in order to correctly return a timezone-aware datetime.  In a future version, this will raise an exception.
  warnings.warn("tzname {tzname} identified but not understood.  "


  ✓ [en] [general] [neutral]
    Title : The Pagoda Puzzle: What Can Save China’s Oldest Wooden Tower?
    Author: Sixth Tone | Tags: china, wooden pagoda, restoration, conservation

[56/500] Source: Sixth Tone China
  URL: https://www.sixthtone.com/news/1018441/China’s Latest Season of ‘Ride the W
  ✓ [en] [entertainment] [negative]
    Title : China’s Latest Season of ‘Ride the Wind’ Goes Viral Due to Poor Perfor
    Author: Marianne Gunnarsson | Tags: china, ride the wind, variety show

[57/500] Source: Sixth Tone China
  URL: https://www.sixthtone.com/news/1018446/How China’s Deaf Delivery Riders Fin
  ✓ [en] [technology] [positive]
    Title : How China's Deaf Delivery Riders Find a New Life in Gig Work
    Author: null | Tags: china, deaf, delivery, riders, gig

[58/500] Source: Sixth Tone China
  URL: https://www.sixthtone.com/news/1018442/Chinese Robot Runner Beats Men’s Hal
  ✓ [en] [technology] [positive]
    Title : Chinese Robot Runner Beats Men's Half-Marathon World Record

EDA summary

In [10]:
df = pd.DataFrame(clean_articles)

print("=" * 50)
print(f"Total articles    : {len(df)}")
print(f"Unique sources    : {df['source'].nunique()}")
print(f"Languages         : {df['language'].value_counts().to_dict()}")
print(f"\n── Category distribution ──")
print(df["category"].value_counts().to_string())
print(f"\n── Sentiment distribution ──")
print(df["sentiment"].value_counts().to_string())
print(f"\n── Articles per source (top 20) ──")
print(df["source"].value_counts().head(20).to_string())
print(f"\n── Null counts ──")
print(df.isnull().sum().to_string())

Total articles    : 252
Unique sources    : 53
Languages         : {'en': 252}

── Category distribution ──
category
technology       74
politics         54
entertainment    27
business         27
sports           18
general          17
health           15
science          12
environment       4
travel            4

── Sentiment distribution ──
sentiment
neutral     170
negative     48
positive     34

── Articles per source (top 20) ──
source
Al Jazeera English          5
What's On Dubai             5
Middle East Eye             5
CGTN World                  5
CGTN Business               5
BBC Technology              5
Xinhua Top News             5
Xinhua China                5
South China Morning Post    5
Sixth Tone China            5
Hong Kong Free Press        5
Nikkei Asia                 5
Asia Times                  5
Japan Times                 5
BBC Top Stories             5
The Verge                   5
TechCrunch AI               5
The Guardian Environment    5
TechCrunch  

In [11]:
from google.colab import files

files.download("news_dataset.csv")
files.download("news_dataset.json")
files.download("news_dataset.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>